# Vector Search Fundamentals: From Embeddings to a Vector Database

Build a small ticket-search system step by step. It starts with manually created toy vectors so the search math is visible, then connects that prototype to real embeddings, vector indexes, metadata filtering, and production vector databases.

## Learning goals

By the end, you should be able to:
- Explain the difference between a vector and an embedding.
- Explain why semantically related text can be found even when wording differs.
- Calculate cosine similarity and rank results.
- Implement simple top-k vector search.
- Explain metadata filtering and brute-force search.
- Describe what a production vector database adds.

> **Important:** The four-number vectors below are hand-crafted teaching examples, not embeddings generated from the ticket text. Their dimensions are given illustrative meanings (`login`, `billing`, `UI`, `performance`) purely to make the math easy to follow. Real embedding dimensions usually do not have such simple human-readable meanings.

## 1. The problem: find related support tickets

Imagine a support team has many tickets. A new ticket says:

> “My authentication token expires too soon.”

A useful search system should find related tickets, such as a login failure or an authentication timeout, even if the wording is different.

Traditional keyword search looks for matching words. Vector search compares numerical representations of content and ranks items by a similarity measure.

For this first exercise, we will skip the embedding model and provide small vectors directly.


In [1]:
import numpy as np

# Each ticket ID maps to:
#   1. a human-readable title
#   2. a four-dimensional toy vector
#
# The dimensions are illustrative:
#   [login/authentication, billing, UI, performance]
#
# These vectors are manually designed for learning. They were NOT
# generated from the ticket titles by an embedding model.
TICKETS = {
    "T-1": ("Login fails after password reset",
            np.array([1.0, 0.0, 0.0, 0.0])),
    "T-2": ("Invoice shows wrong tax",
            np.array([0.0, 1.0, 0.0, 0.0])),
    "T-3": ("Auth token times out early",
            np.array([0.9, 0.1, 0.0, 0.0])),
    "T-4": ("Dashboard button misaligned",
            np.array([0.0, 0.0, 1.0, 0.0])),
    "T-5": ("Board loads slowly with 500 tasks",
            np.array([0.0, 0.0, 0.2, 0.9])),
    "T-6": ("Login page redesign",
            np.array([0.6, 0.0, 0.4, 0.0])),
}

print(f"{len(TICKETS)} tickets seeded")


6 tickets seeded


## 2. What is a vector? What is an embedding?

A **vector** is an ordered list of numbers, such as `[0.9, 0.1, 0.0, 0.0]`.

An **embedding** is a vector produced by a model from an input such as text, an image, or audio. The model is trained so that useful patterns and relationships in the input are represented in the vector space.

A practical text-embedding workflow is:

`ticket text → embedding model → vector → vector search`

Our current notebook starts at the vector stage. That makes the search algorithm easier to learn before introducing a model or API.


## 3. Cosine similarity

Cosine similarity compares the direction of two vectors:

\[
\text{cosine}(a,b)=\frac{a\cdot b}{\|a\|\|b\|}
\]

- The dot product, `a · b`, combines corresponding dimensions.
- The norm, `||a||`, is the vector's length.
- A value near `1` means the vectors point in a similar direction.
- A value near `0` means they are orthogonal (in this example, unrelated directions).
- A value near `-1` means they point in opposite directions.

For non-negative toy vectors like ours, scores will be between 0 and 1. Real embedding models can produce negative cosine similarities too.

If either vector has zero length, the formula would divide by zero, so the function below returns `0.0` as a simple fallback.


In [2]:
def cosine_similarity(a, b):
    """Return cosine similarity between two vectors, rounded to 6 decimals."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    # Length of a vector is its Euclidean norm.
    denominator = np.linalg.norm(a) * np.linalg.norm(b)

    # Avoid division by zero if either vector contains only zeros.
    if denominator == 0:
        return 0.0

    # The @ operator performs the dot product for NumPy arrays.
    return round(float(a @ b / denominator), 6)


# Compare a login ticket with an authentication-timeout ticket.
print("T-1 vs T-3:", cosine_similarity(TICKETS["T-1"][1], TICKETS["T-3"][1]))

# Compare a login ticket with a billing ticket.
print("T-1 vs T-2:", cosine_similarity(TICKETS["T-1"][1], TICKETS["T-2"][1]))


T-1 vs T-3: 0.993884
T-1 vs T-2: 0.0


### Read the result

T-1 and T-3 should have a high score because their toy vectors point in similar directions. T-1 and T-2 should score much lower because they point along different dimensions.

The score is a mathematical comparison of the vectors. It does **not** prove that two tickets are truly related; that depends on the quality of the vectors and the task.


## 4. Create a query vector and score every ticket

A real system would usually turn the user's query text into an embedding. For now, we manually define a query vector that mostly points in the login/authentication direction.


In [3]:
# Toy representation of the query:
# "My authentication token expires too soon."
query_vector = np.array([0.95, 0.05, 0.0, 0.0])

# Calculate a similarity score for every ticket.
scores = {
    ticket_id: cosine_similarity(vector, query_vector)
    for ticket_id, (title, vector) in TICKETS.items()
}

for ticket_id, score in scores.items():
    print(f"{ticket_id}: {score:.6f}")

T-1: 0.998618
T-2: 0.052559
T-3: 0.998314
T-4: 0.000000
T-5: 0.000000
T-6: 0.830900


## 5. Return the top-k nearest tickets

`k` is the number of results requested. The code sorts by:
1. Similarity score, highest first.
2. Ticket ID, alphabetically, to make ties deterministic.

This is called **top-k retrieval**.


In [4]:
def top_k_search(query, k=3):
    """Return the k most similar ticket IDs and their cosine scores."""
    results = []

    for ticket_id, (title, vector) in TICKETS.items():
        score = cosine_similarity(vector, query)
        results.append((ticket_id, score))

    # Sort by descending similarity, then ascending ticket ID for stable ties.
    results.sort(key=lambda item: (-item[1], item[0]))
    return results[:k]


for ticket_id, score in top_k_search(query_vector, k=3):
    title = TICKETS[ticket_id][0]
    print(f"{ticket_id} | {score:.6f} | {title}")


T-1 | 0.998618 | Login fails after password reset
T-3 | 0.998314 | Auth token times out early
T-6 | 0.830900 | Login page redesign


## 6. Build a tiny in-memory vector index

A vector index is a structure that keeps vectors and the information needed to identify them. This simple class stores:
- `ids`: IDs for stored items.
- `vectors`: numerical vectors.
- `metadata`: extra fields, such as a title.

This implementation still checks every stored vector. It is useful for learning, but it is not an advanced approximate-nearest-neighbor index and it is not a full production vector database.


In [5]:
class SimpleVectorIndex:
    """A minimal in-memory vector collection for demonstrating vector search."""

    def __init__(self):
        self.ids = []
        self.vectors = []
        self.metadata = []

    def add(self, item_id, vector, metadata=None):
        """Store one vector and its metadata."""
        vector = np.asarray(vector, dtype=float)

        # Store a copy so later changes to the original array do not
        # silently change the vector held by the index.
        self.ids.append(item_id)
        self.vectors.append(vector.copy())
        self.metadata.append(dict(metadata or {}))

    def search(self, query, k=3):
        """Return top-k matches as (ID, score, metadata) tuples."""
        query = np.asarray(query, dtype=float)
        results = []

        for item_id, vector, metadata in zip(
            self.ids, self.vectors, self.metadata
        ):
            score = cosine_similarity(vector, query)
            results.append((item_id, score, metadata))

        # Highest score first; ID gives stable ordering when scores tie.
        results.sort(key=lambda item: (-item[1], str(item[0])))
        return results[:k]


index = SimpleVectorIndex()

for ticket_id, (title, vector) in TICKETS.items():
    index.add(ticket_id, vector, metadata={"title": title})

for ticket_id, score, metadata in index.search(query_vector, k=3):
    print(f"{ticket_id} | {score:.6f} | {metadata['title']}")


T-1 | 0.998618 | Login fails after password reset
T-3 | 0.998314 | Auth token times out early
T-6 | 0.830900 | Login page redesign


## 7. Metadata filtering

Similarity is not the only requirement. A support agent may want similar tickets that are still open.

Metadata is ordinary structured information stored alongside a vector, for example:
- ticket ID
- title
- status
- team
- creation date

Below, status is stored separately in a dictionary to keep the example straightforward. In a real system, status would normally be part of the ticket record or vector metadata.


In [6]:
STATUS = {
    "T-1": "open",
    "T-2": "open",
    "T-3": "done",
    "T-4": "open",
    "T-5": "open",
    "T-6": "open",
}


def search_filtered(index, query, k=3, status=None):
    """Search only items that match an optional status filter.

    This is a simple pre-filter: records with the wrong status are skipped
    before their similarity score is calculated.
    """
    query = np.asarray(query, dtype=float)
    results = []

    for item_id, vector, metadata in zip(
        index.ids, index.vectors, index.metadata
    ):
        # If a status was requested, skip items with a different status.
        if status is not None and STATUS.get(item_id) != status:
            continue

        score = cosine_similarity(vector, query)
        results.append((item_id, score, metadata))

    results.sort(key=lambda item: (-item[1], str(item[0])))
    return results[:k]


print("Top open tickets:")
for ticket_id, score, metadata in search_filtered(
    index, query_vector, k=3, status="open"
):
    print(f"{ticket_id} | {score:.6f} | {metadata['title']}")


Top open tickets:
T-1 | 0.998618 | Login fails after password reset
T-6 | 0.830900 | Login page redesign
T-2 | 0.052559 | Invoice shows wrong tax


### Pre-filter versus post-filter

This example filters records **before** ranking. Another approach is to retrieve the top-k similar results first and then filter them. These approaches can return different results: post-filtering may leave fewer than `k` matches even when more eligible records exist further down the ranking.

Real vector databases offer different filtering and indexing behaviors, so exact semantics depend on the database and query.


## 8. Compare with NumPy brute-force search

For a small collection, we can place all vectors into one matrix and calculate all cosine similarities in one vectorized operation. This is still an exact comparison against every vector, but NumPy can perform the calculation efficiently.

The matrix below has one row per ticket and one column per vector dimension.


In [7]:
# Preserve the same order for the IDs and the vector matrix.
ticket_ids = list(TICKETS.keys())
vector_matrix = np.stack([TICKETS[ticket_id][1] for ticket_id in ticket_ids])

# Normalize each row to unit length.
row_norms = np.linalg.norm(vector_matrix, axis=1, keepdims=True)
normalized_matrix = np.divide(
    vector_matrix,
    row_norms,
    out=np.zeros_like(vector_matrix, dtype=float),
    where=row_norms != 0,
)

# Normalize the query too. Handle a zero query safely.
query_norm = np.linalg.norm(query_vector)
normalized_query = (
    query_vector / query_norm
    if query_norm != 0
    else np.zeros_like(query_vector, dtype=float)
)

# Dot product between each normalized row and the normalized query
# is the cosine similarity for that row.
all_scores = normalized_matrix @ normalized_query

brute_force_results = sorted(
    zip(ticket_ids, all_scores.round(6)),
    key=lambda item: (-item[1], item[0]),
)[:3]

print("Brute-force top 3:", brute_force_results)
print(
    "Simple index top 3:",
    [(item_id, score) for item_id, score, _ in index.search(query_vector, 3)]
)


Brute-force top 3: [('T-1', 0.998618), ('T-3', 0.998314), ('T-6', 0.8309)]
Simple index top 3: [('T-1', 0.998618), ('T-3', 0.998314), ('T-6', 0.8309)]


### What does this comparison show?

Both approaches should return the same top results for this data. They are both exact searches over all stored vectors. The matrix approach uses vectorized numerical operations; the class-based approach is easier to extend with IDs and metadata.

At large scale, production systems may use specialized indexes (often approximate nearest-neighbor indexes) to avoid comparing the query with every vector. That can improve latency, but may introduce a trade-off between speed, memory, and recall.


## 9. What makes a vector database different?

The `SimpleVectorIndex` class is only a teaching prototype. A production vector database or vector-search system commonly adds capabilities such as:

| Capability | Why it matters |
|---|---|
| Persistent storage | Data remains after the Python process ends |
| Metadata storage and filters | Restrict results by status, tenant, category, date, etc. |
| Indexing | Reduce the work required to find nearest vectors |
| Updates and deletes | Keep vectors synchronized with source records |
| Batch ingestion | Insert many vectors efficiently |
| Scaling and operations | Handle larger datasets, concurrent queries, backups, monitoring, and access control |

Not every product implements every feature in the same way. Some systems are specialized vector databases; others add vector search to a relational database, search engine, or cloud data platform.


## 10. Where embeddings fit in a real application

A typical text retrieval pipeline looks like this:

1. Collect source documents or support tickets.
2. Split long documents into useful chunks when needed.
3. Send each chunk to an embedding model.
4. Store the resulting vector with the original text, ID, and metadata.
5. Embed the user's query with the compatible embedding model.
6. Search for the nearest vectors.
7. Optionally filter, rerank, or apply business rules.
8. Return relevant text or pass it to a language model as context.

For **RAG (retrieval-augmented generation)**, the retrieved text is supplied to the language model so its answer can be grounded in relevant source material. The vector database retrieves candidate content; it does not itself generate the final answer.

The document vectors and query vector must be produced by compatible embedding models and settings. A manually chosen toy vector is not interchangeable with a real model's embedding.


## 11. Exercises

Try these before moving to a real embedding model:

1. Change `k=3` to `k=5`. What changes?
2. Use a billing-oriented query vector such as `[0.0, 1.0, 0.0, 0.0]`. Which ticket ranks first?
3. Add a seventh ticket with a performance-oriented vector and search again.
4. Search only for `status="done"`. How many results are returned?
5. Explain why this class is not a complete production vector database.
6. Explain the difference between a manually designed vector and an embedding generated from text.
7. Explain one reason a production system might use approximate-nearest-neighbor indexing instead of comparing every vector.

### Optional extension

For a next step, replace the manually created vectors with embeddings generated by an actual text-embedding model. Keep the rest of the retrieval pipeline similar, then compare whether the results make sense for natural-language queries.


## Key takeaways

- **Vector:** a list of numbers.
- **Embedding:** a vector produced from content by a model.
- **Similarity metric:** a way to compare vectors; cosine similarity compares direction.
- **Top-k search:** return the highest-ranked matches.
- **Metadata filter:** restrict results using structured fields such as status.
- **Vector index:** a structure that supports vector retrieval.
- **Vector database:** a system that combines vector search with data-management capabilities; exact features vary by product.
- **RAG:** retrieve relevant content first, then give it to a language model as context.

The key teaching progression is: understand the problem → understand vectors → calculate similarity → rank results → add metadata → discuss indexes and databases → introduce real embeddings and RAG.
